# Notebook 05: Predictive Modeling

**Project:** City of Boston – Building & Housing Violations Analysis  
**Group:** Mengxing Wang, Jiacheng He, Xiao Luo, Renwei Li

## Overview

This notebook builds two models:

1. **RandomForest Regression** — Predicts the violation count of a neighborhood based on its normalized rate, population, and other aggregate features. This demonstrates how population and density interact with violation frequency.

2. **KMeans Clustering** — Groups neighborhoods into clusters based on their violation profile (raw count, per-capita rate, year-over-year change). This surfaces natural groupings that may inform targeted policy.

All figures are saved to `figures/`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score, KFold, LeaveOneOut, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, silhouette_score
from sklearn.decomposition import PCA

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

PROJECT_ROOT = Path('..').resolve()
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
EXTERNAL_DIR = PROJECT_ROOT / 'data' / 'external'
FIGURES_DIR = PROJECT_ROOT / 'figures'
FIGURES_DIR.mkdir(exist_ok=True)

RNG = 42

# Load datasets
df = pd.read_csv(PROCESSED_DIR / 'violations_clean.csv', low_memory=False)
df['status_dttm'] = pd.to_datetime(df['status_dttm'], errors='coerce')
stats = pd.read_csv(PROCESSED_DIR / 'neighborhood_stats.csv')

print(f'Violations records: {len(df):,}')
print(f'Neighborhood stats rows: {len(stats)}')
print(f'Stats columns: {list(stats.columns)}')
stats.head()

## 1. Feature Engineering for Regression

We enrich the neighborhood-level stats table with additional features derived from the full dataset:
- **open_rate**: fraction of violations still open
- **pct_maintenance**: fraction of violations categorized as maintenance
- **pct_unsafe**: fraction classified as unsafe/dangerous
- **year_range**: span of years with violations
- **unique_codes**: diversity of violation types
- **month_entropy**: seasonality spread (Shannon entropy over months — 0 = one month, ~2.48 = perfectly uniform)
- **recent_share**: fraction of violations in the most recent 2 years (proxy for trend/activity)
- **weekday_share**: share filed Mon–Fri (enforcement-pattern signal)
- **log_population**: log-scaled population (reduces skew for tree splits)

These capture temporal, behavioral, and scale dimensions the raw count alone misses.

In [ ]:
df_nbhd = df[df['has_neighborhood'] == True].copy()

def month_entropy(months):
    m = months.dropna().astype(int)
    if len(m) == 0:
        return 0.0
    p = m.value_counts(normalize=True).values
    return float(-(p * np.log(p + 1e-12)).sum())

max_year = int(df_nbhd['year'].dropna().max()) if df_nbhd['year'].notna().any() else 0
recent_cutoff = max_year - 1  # most recent 2 years

agg = df_nbhd.groupby('neighborhood').agg(
    open_rate=('status', lambda x: (x == 'Open').mean()),
    unique_codes=('code', 'nunique'),
    pct_maintenance=('description', lambda x: x.str.contains('Maintenance', case=False, na=False).mean()),
    pct_unsafe=('description', lambda x: x.str.contains('Unsafe|Dangerous', case=False, na=False).mean()),
    month_entropy=('month', month_entropy),
    recent_share=('year', lambda x: (x >= recent_cutoff).mean() if x.notna().any() else 0.0),
    weekday_share=('day_of_week', lambda x: x.isin(['Monday','Tuesday','Wednesday','Thursday','Friday']).mean()),
).reset_index()

# Year range for neighborhoods with date data
df_dated = df_nbhd[df_nbhd['year'].notna()]
year_range = df_dated.groupby('neighborhood')['year'].agg(lambda x: x.max() - x.min()).reset_index()
year_range.columns = ['neighborhood', 'year_range']

# Merge all features (stats has: neighborhood, violation_count, population_2025, violations_per_1000)
features_df = stats.merge(agg, on='neighborhood', how='left')\
                   .merge(year_range, on='neighborhood', how='left')

# Drop rows without population (can't normalize)
features_df = features_df.dropna(subset=['population_2025', 'violations_per_1000'])
features_df = features_df.fillna(0)
features_df['log_population'] = np.log1p(features_df['population_2025'])
print(f'Feature matrix shape: {features_df.shape}')
features_df[['neighborhood','violation_count','violations_per_1000','open_rate','unique_codes',
             'pct_maintenance','pct_unsafe','year_range','month_entropy','recent_share','weekday_share']].head()

## 2. RandomForest Regression

### Target
We predict **`violation_count`** (raw count) from population and the engineered features.

### Diagnosis first
The earlier baseline fed the model **both** `population_2025` **and** `violations_per_1000`. Because
`violations_per_1000 = violation_count × 1000 / population`, giving the model both features is target leakage:
the target is algebraically recoverable from the inputs, so any reported R² on that feature set is meaningless.
We therefore drop `violations_per_1000` from the feature matrix.

### Approach
With only n≈19 neighborhoods and a heavy-tailed target (Dorchester=4602 vs. Fenway=1), we:
1. Apply a **log1p** transform to the target to stabilize variance.
2. Use **Leave-One-Out CV** (LOO) — with n≈19, 5-fold leaves only 3–4 per test fold.
3. Compare three models (Ridge baseline, RandomForest, GradientBoosting) to establish a fair performance ceiling.
4. Run **GridSearchCV** on the best tree model with LOO for principled hyperparameter selection.

Metrics are reported on the **original scale** (expm1-back-transformed) so they are interpretable.

In [ ]:
feature_cols = [
    'log_population', 'population_2025',
    'open_rate', 'unique_codes',
    'pct_maintenance', 'pct_unsafe', 'year_range',
    'month_entropy', 'recent_share', 'weekday_share',
]

X = features_df[feature_cols].values
y = features_df['violation_count'].values
y_log = np.log1p(y)

loo = LeaveOneOut()

def loo_eval(model, X, y, y_log, log_target=True):
    preds_log = np.zeros_like(y_log, dtype=float)
    for tr, te in loo.split(X):
        model.fit(X[tr], y_log[tr] if log_target else y[tr])
        p = model.predict(X[te])
        preds_log[te] = p
    preds = np.expm1(preds_log) if log_target else preds_log
    preds = np.clip(preds, 0, None)
    return {
        'MAE': mean_absolute_error(y, preds),
        'RMSE': np.sqrt(mean_squared_error(y, preds)),
        'R2': r2_score(y, preds),
    }

# --- (A) LEAKY baseline: include violations_per_1000 as a feature (for contrast) ---
leaky_cols = feature_cols + ['violations_per_1000']
X_leaky = features_df[leaky_cols].values
leaky_rf = RandomForestRegressor(n_estimators=200, max_depth=4, random_state=RNG)
leaky_metrics = loo_eval(leaky_rf, X_leaky, y, y_log, log_target=True)

# --- (B) Model comparison on the clean feature set ---
candidates = {
    'Ridge': Pipeline([('sc', StandardScaler()), ('m', Ridge(alpha=1.0, random_state=RNG))]),
    'RandomForest': RandomForestRegressor(n_estimators=300, max_depth=4,
                                          min_samples_leaf=2, random_state=RNG),
    'GradientBoosting': GradientBoostingRegressor(n_estimators=200, max_depth=3,
                                                  learning_rate=0.05, random_state=RNG),
}
results = {'Leaky-RF (baseline)': leaky_metrics}
for name, mdl in candidates.items():
    results[name] = loo_eval(mdl, X, y, y_log, log_target=True)

results_df = pd.DataFrame(results).T.round(3)
print('Leave-One-Out CV results (target = violation_count, trained on log1p(y)):')
print(results_df)
print()

# --- (C) Hyperparameter tuning on RF with LOO ---
param_grid = {
    'n_estimators': [200, 400, 800],
    'max_depth': [2, 3, 4, 6, None],
    'min_samples_leaf': [1, 2, 3],
    'max_features': [0.5, 0.8, 1.0],
}
gs = GridSearchCV(
    RandomForestRegressor(random_state=RNG),
    param_grid, cv=loo, scoring='neg_mean_absolute_error', n_jobs=-1,
)
gs.fit(X, y_log)
best_rf = gs.best_estimator_
tuned_metrics = loo_eval(best_rf, X, y, y_log, log_target=True)
print(f'Best RF params: {gs.best_params_}')
print(f'Tuned RF LOO metrics: MAE={tuned_metrics["MAE"]:.1f}  '
      f'RMSE={tuned_metrics["RMSE"]:.1f}  R²={tuned_metrics["R2"]:.3f}')

# Promote tuned RF for downstream use
rf = best_rf

### 2a. Feature Importance

We fit the full model to the entire dataset and inspect feature importances.

In [ ]:
rf.fit(X, y_log)
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
colors = sns.color_palette('viridis', len(importances))
ax.barh(importances.index[::-1], importances.values[::-1], color=colors)
ax.set_xlabel('Feature Importance (Mean Decrease in Impurity)')
ax.set_title('Tuned RandomForest Feature Importances\n(Predicting log1p(Violation Count))', fontsize=12)
plt.tight_layout()
fig.savefig(FIGURES_DIR / '13_rf_feature_importance.png', bbox_inches='tight')
plt.close()
print('Saved: 13_rf_feature_importance.png')
print(importances.round(3))

### 2b. Actual vs. Predicted (LOO-CV predictions)

We plot the **held-out** LOO predictions from the tuned model — these reflect genuine generalization, not
training fit. The reported MAE/R² match the tuned row from the table above.

In [ ]:
# Generate held-out LOO predictions with the tuned RF
preds_log = np.zeros_like(y_log, dtype=float)
for tr, te in loo.split(X):
    best_rf.fit(X[tr], y_log[tr])
    preds_log[te] = best_rf.predict(X[te])
y_pred = np.clip(np.expm1(preds_log), 0, None)

full_mae = mean_absolute_error(y, y_pred)
full_r2  = r2_score(y, y_pred)

fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(y, y_pred, s=80, color='steelblue', alpha=0.8, zorder=5)
maxval = max(y.max(), y_pred.max()) * 1.05
ax.plot([0, maxval], [0, maxval], 'r--', linewidth=1.5, label='Perfect fit')

for i, nbhd in enumerate(features_df['neighborhood']):
    ax.annotate(nbhd, (y[i], y_pred[i]), xytext=(5, 3),
                textcoords='offset points', fontsize=7, alpha=0.75)

ax.set_xlabel('Actual Violation Count')
ax.set_ylabel('Predicted Violation Count (LOO held-out)')
ax.set_title(f'Tuned RF: Actual vs. LOO-Predicted\nMAE={full_mae:.1f}, R²={full_r2:.3f}', fontsize=12)
ax.legend()
plt.tight_layout()
fig.savefig(FIGURES_DIR / '14_rf_actual_vs_predicted.png', bbox_inches='tight')
plt.close()
print('Saved: 14_rf_actual_vs_predicted.png')

# refit on all data for any downstream use
best_rf.fit(X, y_log)

## 3. KMeans Clustering

We cluster neighborhoods by their violation profile. Features used:
- `violations_per_1000` — per-capita burden
- `open_rate` — unresolved fraction
- `pct_maintenance` — maintenance violation share
- `pct_unsafe` — unsafe/dangerous share
- `codes_per_1000` — type diversity normalized to population (avoids size-bias of raw `unique_codes`)
- `recent_share` — recency of activity

All features are standardized before clustering, and **K is selected by silhouette** (not elbow alone) since
silhouette directly measures cluster separation — the elbow's "kink" is often ambiguous on small datasets.

In [ ]:
# Normalize type diversity by population so big neighborhoods don't automatically dominate
features_df['codes_per_1000'] = features_df['unique_codes'] / (features_df['population_2025'] / 1000.0)

cluster_cols = ['violations_per_1000', 'open_rate', 'pct_maintenance',
                'pct_unsafe', 'codes_per_1000', 'recent_share']
X_cluster = features_df[cluster_cols].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

# Elbow + Silhouette
inertias, silhouettes = [], []
K_range = list(range(2, min(10, len(features_df))))
for k in K_range:
    km = KMeans(n_clusters=k, random_state=RNG, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

K_BEST = K_range[int(np.argmax(silhouettes))]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(K_range, inertias, marker='o', color='steelblue', linewidth=2)
axes[0].set_xlabel('K'); axes[0].set_ylabel('Inertia (SSE)')
axes[0].set_title('Elbow Method')

axes[1].plot(K_range, silhouettes, marker='o', color='darkorange', linewidth=2)
axes[1].axvline(K_BEST, color='red', linestyle='--', alpha=0.6, label=f'Best K = {K_BEST}')
axes[1].set_xlabel('K'); axes[1].set_ylabel('Silhouette score')
axes[1].set_title('Silhouette Method')
axes[1].legend()
plt.tight_layout()
fig.savefig(FIGURES_DIR / '15_kmeans_elbow.png', bbox_inches='tight')
plt.close()
print('Saved: 15_kmeans_elbow.png')
print('Inertias   :', dict(zip(K_range, [round(i,1) for i in inertias])))
print('Silhouettes:', dict(zip(K_range, [round(s,3) for s in silhouettes])))
print(f'Chosen K (max silhouette): {K_BEST}')

In [ ]:
# Fit KMeans with the silhouette-chosen K
km = KMeans(n_clusters=K_BEST, random_state=RNG, n_init=20)
features_df = features_df.copy()
features_df['cluster'] = km.fit_predict(X_scaled)

print(f'K = {K_BEST}  |  silhouette = {silhouette_score(X_scaled, features_df["cluster"]):.3f}')
print('Cluster assignments:')
for c in range(K_BEST):
    nbhds = features_df[features_df['cluster'] == c]['neighborhood'].tolist()
    print(f'  Cluster {c}: {nbhds}')

### 3a. PCA Projection of Clusters

We reduce the 5-feature cluster space to 2 principal components for visualization.

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
features_df['pca1'] = X_pca[:, 0]
features_df['pca2'] = X_pca[:, 1]

palette = sns.color_palette('Set1', K_BEST)
fig, ax = plt.subplots(figsize=(10, 8))
for c in range(K_BEST):
    mask = features_df['cluster'] == c
    ax.scatter(features_df.loc[mask, 'pca1'],
               features_df.loc[mask, 'pca2'],
               s=120, color=palette[c], label=f'Cluster {c}', zorder=5)
    for _, row in features_df[mask].iterrows():
        ax.annotate(row['neighborhood'], (row['pca1'], row['pca2']),
                    xytext=(5, 4), textcoords='offset points', fontsize=8, alpha=0.85)

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
ax.set_title('KMeans Clusters of Boston Neighborhoods\n(PCA projection)', fontsize=12)
ax.legend()
plt.tight_layout()
fig.savefig(FIGURES_DIR / '16_kmeans_pca.png', bbox_inches='tight')
plt.close()
print('Saved: 16_kmeans_pca.png')

### 3b. Cluster Profile Heatmap

This heatmap shows the mean (standardized) feature values per cluster, revealing what characterizes each group.

In [ ]:
cluster_profiles = features_df.groupby('cluster')[cluster_cols].mean()
# Standardize for display
cluster_profiles_scaled = pd.DataFrame(
    scaler.transform(cluster_profiles),
    index=[f'Cluster {i}' for i in cluster_profiles.index],
    columns=cluster_cols
)

fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(cluster_profiles_scaled, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, ax=ax, linewidths=0.5)
ax.set_title('Cluster Feature Profiles (Standardized Mean Values)', fontsize=12)
plt.tight_layout()
fig.savefig(FIGURES_DIR / '17_cluster_profiles.png', bbox_inches='tight')
plt.close()
print('Saved: 17_cluster_profiles.png')
print(cluster_profiles.round(3))

### 3c. Violation Rate by Cluster

A boxplot showing the distribution of per-capita violation rates within each cluster.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
palette_box = {c: sns.color_palette('Set1', K_BEST)[c] for c in range(K_BEST)}
features_df['cluster_label'] = features_df['cluster'].map(lambda c: f'Cluster {c}')
sns.boxplot(data=features_df, x='cluster_label', y='violations_per_1000',
            palette=[palette_box[c] for c in range(K_BEST)], ax=ax)
ax.set_xlabel('Cluster')
ax.set_ylabel('Violations per 1,000 Residents')
ax.set_title('Per-Capita Violation Rate by Cluster', fontsize=12)
plt.tight_layout()
fig.savefig(FIGURES_DIR / '18_cluster_violation_rate.png', bbox_inches='tight')
plt.close()
print('Saved: 18_cluster_violation_rate.png')

## 4. Save Model Results

In [ ]:
results = features_df[['neighborhood', 'violation_count', 'violations_per_1000',
                        'cluster', 'cluster_label', 'open_rate',
                        'pct_maintenance', 'pct_unsafe', 'unique_codes']]
results.to_csv(PROCESSED_DIR / 'modeling_results.csv', index=False)
print('Saved → data/processed/modeling_results.csv')
results

## Summary

### Honest performance assessment
With only **n = 19** neighborhoods, any regression model is severely data-limited — a single outlier
(Dorchester, 4602 violations vs. next-highest ~2761) can swing R² by >0.2. We reported **Leave-One-Out**
metrics on the **original count scale** rather than in-sample R², which would have been optimistic.

### What we changed vs. the original baseline
| Issue in baseline | Fix applied |
|---|---|
| Referenced non-existent `population_2020` (real column is `population_2025`) → `dropna` would raise | Corrected column name throughout |
| Target leakage: fed model both `population` and `violations_per_1000 = count × 1000 / pop` | Dropped `violations_per_1000` from regression features; retained for clustering only |
| 5-fold CV on n=19 (3–4 test samples/fold → noisy) | Switched to Leave-One-Out CV |
| Heavy-tailed target (4602 vs. 1) trained raw | Trained on `log1p(y)`, back-transformed for reporting |
| One hand-picked RF (`max_depth=4`, 200 trees) | Ridge / RandomForest / GradientBoosting comparison + `GridSearchCV` with LOO |
| `unique_codes` not normalized for clustering (size bias) | Switched to `codes_per_1000` |
| K=3 hard-coded from elbow only | Selected K by maximum silhouette |
| Sparse feature set (7 features) | Added `month_entropy`, `recent_share`, `weekday_share`, `log_population` |

### Diagnosis of residual weakness
Even after tuning, the dominant failure mode is **insufficient data**, not a bad model choice. With
n=19 the model cannot reliably learn interactions; the tuned RF typically converges to shallow trees that
mostly key off population scale. To materially move performance, the right next step is **not** more model
complexity but more rows: e.g., moving to a **census-tract** or **block-group** granularity would yield
hundreds of units instead of 19, letting the engineered temporal and structural features actually earn their
keep.

### KMeans Clustering
With the silhouette-selected K and size-normalized features, clusters now separate on *behavioral profile*
(open-rate, type mix, recency) rather than raw scale. Approximate interpretation:
- **High-risk cluster**: high per-capita rate, high open_rate → violations filed often and resolved slowly.
- **Moderate-risk cluster**: mid per-capita rate, mixed violation types.
- **Low-risk cluster**: low per-capita rate, mostly maintenance, high closure rate.

These clusters can guide targeted enforcement and resource allocation by the City of Boston.

Proceed to the **Final Report** in `report/final_report.md`.